deep diving into: F.scaled_dot_product_attention

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken

In [12]:
class MultiHead(nn.Module):
	def __init__(self, C, nheads):
		super().__init__()
		self.nh = nheads
		self.d = C // nheads

		# W_qkv = [ W_q | W_k | W_v ]   # shape (C, 3C)
		self.QKV = nn.Linear(C, 3*C, bias=False)
		self.W0 = nn.Linear(C, C, bias=False)

	def forward(self, x):
		B, T, C = x.shape

		# (C, 3C) * (B, T, C) -> (B, T, 3C) -> split -> (B, T, C), (B, T, C), (B, T, C)
		# [ W_q | W_k | W_v ] * x 
		q, k, v = self.QKV(x).split(C, dim=-1)

		# tranpose because data is in wrong order
		# (B, T, C) -> (B, nh, T, C/nh)
		# change dims for nh heads.
		# "each head" makes a decision and we average their answers
		# each head gets a d-sized slice of q, k, v. 
		# each d-slice is a learned mix of the embedding from QKV
		# so each head literally sees a different view of the embedding and can make a different decision
		q, k, v = (t.view(B, T, self.nh, self.d).transpose(1, 2) for t in (q, k, v))   

		# add triangular mask for causal attention
		# do the attention = softmax(QK^T / sqrt(d)) V
		# but on a kernel that fuses the operations and uses less memory
		attn = F.scaled_dot_product_attention(q, k, v, is_causal=True)
		with torch.profiler.profile() as prof:
			F.scaled_dot_product_attention(q, k, v, is_causal=True)
		print(prof.key_averages().table())

		# concat heads
		# (B, nh, T, d) -> (B, T, nh, d) -> (B, T, C)
		# contiguous() copies the tensor into a fresh block of memory laid out in its current logical order.
		attn = attn.transpose(1, 2).contiguous().view(B, T, C)

		return self.W0(attn)

In [13]:
enc = tiktoken.get_encoding("gpt2")
sample_text = """
Elon Reeve Musk (born June 28, 1971) is a businessman, industrialist, and former public official who is the chief executive officer (CEO) and largest shareholder of Tesla and SpaceX. Musk has been the wealthiest person in the world since 2025, and briefly became the only trillionaire (in terms of US dollars) in June 2026; as of September 2026, Forbes estimates his net worth to be US$964 billion.
Born into the wealthy Musk family in Pretoria, South Africa, Musk emigrated in 1989 to Canada; he has Canadian citizenship since his mother was born there. He received bachelor's degrees in 1997 from the University of Pennsylvania before moving to California to pursue business ventures. In 1995, Musk co-founded Zip2, a web software company. Following its sale in 1999, he co-founded X.com, an e-commerce payment system that merged with Confinity in March 2000 to form PayPal, which was acquired by eBay in 2002. Musk also became an American citizen in 2002.
In 2002, Musk founded and became CEO and chief engineer of SpaceX, a space technology company; the company has since led innovations in reusable rockets and commercial spaceflight. Musk joined Tesla as an early investor in 2004 and became its CEO and product architect in 2008; it has since become a leader in electric vehicles. In 2015, Musk co-founded OpenAI to advance artificial intelligence (AI) research, but later left; his growing discontent with the organization's direction and leadership in the AI boom in the 2020s led him to establish xAI, which became a subsidiary of SpaceX in 2026. In 2022, he acquired Twitter, a social networking service, and made significant changes, including rebranding it as X in 2023. His other businesses include Neuralink, a neurotechnology company that he co-founded in 2016, and the Boring Company, a tunneling company that he founded in 2017. In November 2025, Tesla approved a pay package worth $1 trillion for Musk, which he is to receive over 10 years if certain milestones are met, such as achieving a market capitalization of $8.5 trillion. Musk became the first US-dollar trillionaire upon the initial public offering of SpaceX in June 2026, though a SpaceX stock collapse brought him under the threshold by the next month.
Musk was the largest donor in the 2024 U.S. presidential election, where he supported Donald Trump. After Trump was inaugurated in January 2025, Musk was Senior Advisor to the President and the de facto head of the Department of Government Efficiency (DOGE). Musk left the Trump administration in May 2025 and returned to managing his companies; shortly thereafter he had a public feud with Trump.
Musk is a supporter of global far-right politics, figures, and political parties. His political activities, statements and views have made him a polarizing figure. He has been criticized for making unscientific and misleading statements, including spreading COVID-19 misinformation, promoting conspiracy theories, and affirming antisemitic, white nationalist, racist, and transphobic comments. His acquisition of Twitter was controversial because, following his pledge to decrease censorship, there was an increase in hate speech and misinformation on the service. His role in the second Trump administration attracted public backlash, particularly in response to DOGE and its cuts to the US Agency for International Development (USAID).
"""
encoded_text = enc.encode(sample_text)
encoded_tensor = torch.tensor(encoded_text).unsqueeze(0)
encoded_tensor.shape  # should be (1, T) where T is the length of the encoded text

torch.Size([1, 690])

In [14]:
model = MultiHead(C=768, nheads=12)
x = torch.randn(1, encoded_tensor.shape[1], 768)  # (B, T, C) where B=1, T=length of encoded text, C=768
output = model(x)
output.shape  # should be (1, T, 768) where T is the length of the encoded text

-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                 Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                   aten::scaled_dot_product_attention         0.11%      20.422us       100.00%      18.161ms      18.161ms             1  
    aten::_scaled_dot_product_flash_attention_for_cpu        99.40%      18.053ms        99.89%      18.140ms      18.140ms             1  
                                     aten::empty_like         0.04%       8.024us         0.22%      39.189us      39.189us             1  
                                  aten::empty_strided         0.17%      31.165us         0.17%      31.165us      31.165us             1  
                    

torch.Size([1, 690, 768])